## Controller

Goal: Implement controller component of world models

In [7]:
import torch
from torch import nn

from world_models.models.mdn_rnn import MDNRNN


class Controller(nn.Module):
    def __init__(self, latent_dim=32, hidden_dim=256):
        super().__init__()
        self.linear = nn.Linear(latent_dim + hidden_dim, 3)

    def forward(self, z, h):
        # z: [B, latent_dim]
        # h: [B, hidden_dim]
        features = torch.cat([z, h], dim=-1)
        raw = self.linear(features)

        steering = torch.tanh(raw[:, 0:1])
        gas = (torch.tanh(raw[:, 1:2]) + 1) / 2
        brake = torch.tanh(raw[:, 2:3]).clamp_min(0)

        return torch.cat([steering, gas, brake], dim=-1)

In [8]:
@torch.no_grad()
def agent_step(z, state, memory, controller):
    if state is None:
        # Beginning of an episode: no previous memory.
        h = z.new_zeros(z.shape[0], memory.lstm.hidden_size)
    else:
        # state = (hidden_state, cell_state)
        h = state[0][-1]  # [B, hidden_dim]

    action = controller(z, h)

    _, _, _, next_state = memory(
        z.unsqueeze(1),       # [B, 1, latent_dim]
        action.unsqueeze(1),  # [B, 1, action_dim]
        state=state,
    )

    return action, next_state

In [9]:
torch.manual_seed(0)

memory = MDNRNN(
    latent_dim=32,
    action_dim=3,
    hidden_dim=256,
    num_components=5,
).eval()

controller = Controller().eval()

state = None  # Reset once at the beginning of each episode.

for t in range(2):
    z = torch.randn(1, 32)
    action, state = agent_step(z, state, memory, controller)

    print(f"Step {t}: action = {action.squeeze(0).tolist()}")

print("Hidden shape:", state[0].shape)  # [1, 1, 256]
print("Cell shape:", state[1].shape)    # [1, 1, 256]
print(
    "Controller parameters:",
    sum(p.numel() for p in controller.parameters()),
)  # 867

Step 0: action = [0.003618634305894375, 0.5615862607955933, 0.18155668675899506]
Step 1: action = [-0.027960803359746933, 0.4560697674751282, 0.0]
Hidden shape: torch.Size([1, 1, 256])
Cell shape: torch.Size([1, 1, 256])
Controller parameters: 867


In [10]:
@torch.no_grad()
def rollout(
    env,
    encode_observation,
    memory,
    controller,
    seed=0,
    max_steps=1000,
):
    """Evaluate one episode using a Gymnasium-style environment."""
    memory.eval()
    controller.eval()

    observation, info = env.reset(seed=seed)
    state = None
    total_reward = 0.0
    steps = 0

    for _ in range(max_steps):
        # Must return [1, latent_dim] on the models' device.
        z = encode_observation(observation)

        action, state = agent_step(
            z, state, memory, controller
        )

        observation, reward, terminated, truncated, info = env.step(
            action.squeeze(0).cpu().numpy()
        )

        total_reward += float(reward)
        steps += 1

        if terminated or truncated:
            break

    return {"reward": total_reward, "steps": steps}

In [11]:
import numpy as np
from PIL import Image

from world_models.models.vae import VAE

# Fresh weights for checking the full pipeline.
vae = VAE().eval()


@torch.no_grad()
def encode_observation(observation):
    # Environment image: [H, W, 3], uint8.
    image = Image.fromarray(observation).resize(
        (64, 64),
        resample=Image.Resampling.BILINEAR,
    )

    pixels = np.array(image, dtype=np.float32) / 255.0
    device = next(vae.parameters()).device

    x = (
        torch.from_numpy(pixels)
        .permute(2, 0, 1)  # [3, 64, 64]
        .unsqueeze(0)     # [1, 3, 64, 64]
        .to(device)
    )

    mu, logvar = vae.encode(x)
    return vae.reparameterize(mu, logvar)  # [1, 32]

In [12]:
import gymnasium as gym

# Keep all three models on the same device for this initial run.
vae = vae.cpu().eval()
memory = memory.cpu().eval()
controller = controller.cpu().eval()

env = gym.make(
    "CarRacing-v3",
    continuous=True,
    domain_randomize=False,
)

try:
    torch.manual_seed(0)  # Controls VAE sampling.
    result = rollout(
        env=env,
        encode_observation=encode_observation,
        memory=memory,
        controller=controller,
        seed=0,          # Controls track generation.
        max_steps=200,
    )

    print(f"Steps: {result['steps']}")
    print(f"Total reward: {result['reward']:.2f}")
finally:
    env.close()

Steps: 183
Total reward: -80.58
